# Kusaal sentence alignment — Stage 1 + Stage 2
Before running: **Runtime → Change runtime type → T4 GPU**, and add your HF token in **Secrets** (key icon, label `HF_TOKEN`, notebook access ON).

Reads `kusaal_mt/wiki_pairs/article_pairs.jsonl` from Drive; writes everything to `kusaal_mt/stage2/`. Safe to re-run after a disconnect — translation resumes from its checkpoint.

## Setup

In [ ]:
# ---- setup: Drive, deps, HF token ----------------------------------
%pip install -q sentence-transformers sacrebleu

import collections, csv, json, os, re, time, unicodedata
from google.colab import drive, userdata

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/kusaal_mt'
IN_PATH = f'{BASE}/wiki_pairs/article_pairs.jsonl'
OUT_DIR = f'{BASE}/stage2'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(IN_PATH), f'missing {IN_PATH} - is Drive mounted?'

hf_token = None
for name in ['HF_TOKEN', 'HUGGINGFACE_TOKEN', 'HF_API_TOKEN']:
    try:
        hf_token = userdata.get(name)
        print('using secret:', name)
        break
    except Exception:
        continue
if not hf_token:
    raise RuntimeError('Add your HF token in Colab Secrets (key icon, label HF_TOKEN) '
                       'and switch on "Notebook access".')

import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## Stage 1 functions (identical to the local run)

In [ ]:
# --- cleaning ---------------------------------------------------

# Model training data uses straight apostrophe U+0027 exclusively;
# the wiki mixes saltillo/curly variants. Fold them all down.
APOSTROPHE_MAP = str.maketrans({
    "\uA78C": "'", "\uA78B": "'", "\u02BC": "'", "\u02B9": "'",
    "\u2019": "'", "\u2018": "'", "\u2032": "'",
    "\u201C": '"', "\u201D": '"', "\u00A0": " ",
})

JUNK_SECTIONS = {
    "references", "external links", "see also", "sources", "notes",
    "further reading", "bibliography", "footnotes", "gallery",
    "citations", "works cited",
}

EN_ABBREVS = [
    "Mr.", "Mrs.", "Ms.", "Dr.", "Prof.", "St.", "No.", "Jr.", "Sr.",
    "vs.", "etc.", "e.g.", "i.e.", "cf.", "ca.", "approx.", "Rev.",
    "Hon.", "Lt.", "Gen.", "Col.", "Capt.", "Sgt.", "a.m.", "p.m.",
    "Op.", "pp.", "Vol.", "Ltd.", "Inc.", "Co.",
]

DOT = "\x00"  # placeholder protecting non-boundary periods

deglue_count = 0


def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = text.translate(APOSTROPHE_MAP)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return text


def deglue_kus(text):
    """Fix stripped-link artifacts like 'DistrictJohn Atta Mills':
    split lowercase->uppercase glue when >=3 lowercase letters precede
    (leaves McDonald/YouTube-style names alone). Kusaal side only."""
    global deglue_count
    fixed, n = re.subn(r"([a-zɛɔʋŋ']{3})([A-ZƐƆƲŊ])", r"\1 \2", text)
    deglue_count += n
    return fixed


def protect_dots(line, lang):
    if lang == "en":
        for ab in EN_ABBREVS:
            line = line.replace(ab, ab.replace(".", DOT))
    line = re.sub(r"\b([A-ZƐƆƲŊ])\.", r"\1" + DOT, line)   # initials: J. J. Rawlings
    line = re.sub(r"(?<=\d)\.(?=\d)", DOT, line)            # decimals: 3.5
    return line


def is_sentence(s):
    if len(s) < 15 or len(s) > 1200:
        return False
    if len(re.findall(r"[^\W\d_]", s)) < len(s) * 0.5:      # mostly letters
        return False
    if len(s.split()) < 3:
        return False
    return s[-1] in ".!?\"')"


def split_sentences(text, lang):
    """Clean one article and return its list of sentences, in order."""
    sents, seen = [], set()
    section = ""
    for raw in text.split("\n"):
        line = raw.strip()
        if not line:
            continue
        m = re.match(r"^=+\s*(.+?)\s*=+$", line)
        if m:
            section = m.group(1).lower()
            continue
        if section in JUNK_SECTIONS:
            continue
        if line[0] in "*-•|#":
            continue
        line = protect_dots(line, lang)
        for part in re.split(r"(?<=[.!?])\s+", line):
            part = part.replace(DOT, ".").strip()
            part = re.sub(r"\s+", " ", part)
            if is_sentence(part) and part not in seen:
                seen.add(part)
                sents.append(part)
    return sents


# --- anchors ----------------------------------------------------

def anchor_tokens(sent):
    """Split a sentence into hard anchors (numbers, capitalized words)
    and soft anchors (lowercase words >= 4 chars). Hard tokens shared
    between a Kusaal and an English sentence are almost always proper
    names, years, or figures; shared soft tokens are English loanwords
    ('security', 'university'), common in Kusaal wiki prose."""
    hard, soft = set(), set()
    for t in re.findall(r"[^\W_]+", sent):
        if any(c.isdigit() for c in t):
            if len(t) >= 2:                 # keep 14, 1960; drop bare 0-9
                hard.add(t)
        elif t[:1].isupper() and len(t) >= 3:
            hard.add(t)
        elif len(t) >= 4:
            soft.add(t)
    return hard, soft


def align_article(kus_sents, en_sents):
    """Anchor-match one article. Returns (rows, unmatched_indices)."""
    kus_hard, kus_soft = zip(*(anchor_tokens(s) for s in kus_sents))
    en_hard, en_soft = zip(*(anchor_tokens(s) for s in en_sents))

    df_kus = collections.Counter(a for s in kus_hard for a in s)
    df_en = collections.Counter(a for s in en_hard for a in s)
    dfs_kus = collections.Counter(a for s in kus_soft for a in s)
    dfs_en = collections.Counter(a for s in en_soft for a in s)

    def score(ki, ei):
        shared = kus_hard[ki] & en_hard[ei]
        shared_soft = kus_soft[ki] & en_soft[ei]
        if not shared and not shared_soft:
            return 0.0, shared, shared_soft
        s = sum(1 / df_kus[a] + 1 / df_en[a] for a in shared)
        s += 0.5 * sum(1 / dfs_kus[a] + 1 / dfs_en[a] for a in shared_soft)
        k_rel = ki / max(len(kus_sents) - 1, 1)
        e_rel = ei / max(len(en_sents) - 1, 1)
        s += 0.3 * (1 - abs(k_rel - e_rel))
        return s, shared, shared_soft

    # best English candidate for every Kusaal sentence (and reverse)
    best_for_en = {}
    scored = []
    for ki in range(len(kus_sents)):
        cands = []
        for ei in range(len(en_sents)):
            s, shared, shared_soft = score(ki, ei)
            if s > 0:
                cands.append((s, ei, shared, shared_soft))
        cands.sort(key=lambda c: (c[0], -c[1]))
        cands.reverse()
        scored.append(cands)
        if cands:
            s, ei = cands[0][0], cands[0][1]
            if s > best_for_en.get(ei, (0,))[0]:
                best_for_en[ei] = (s, ki)

    rows, unmatched = [], []
    for ki, cands in enumerate(scored):
        if not cands:
            unmatched.append(ki)
            continue
        s1, ei, shared, shared_soft = cands[0]
        s2 = cands[1][0] if len(cands) > 1 else 0.0
        margin = s1 - s2
        mutual = best_for_en.get(ei, (0, -1))[1] == ki
        distinctive = any(df_kus[a] <= 2 and df_en[a] <= 2 for a in shared)
        # a lone shared name ("Adam", "Europe") is not enough evidence:
        # confident also needs either 2+ hard anchors or a strong score
        strong = len(shared) >= 2 or s1 >= 2.2
        if distinctive and mutual and strong and (len(cands) == 1 or margin >= 0.5):
            tier = "confident"
        else:
            tier = "unsure"
        anchors = " ".join(sorted(shared))
        if shared_soft:
            anchors += f" (+{len(shared_soft)} loanwords)"
        rows.append({
            "tier": tier,
            "score": round(s1, 3),
            "margin": round(margin, 3),
            "mutual": "yes" if mutual else "no",
            "anchors": anchors.strip(),
            "kus_pos": ki,
            "en_pos": ei,
            "kus_sentence": kus_sents[ki],
            "en_sentence": en_sents[ei],
        })
    return rows, unmatched


## Stage 1 — clean + anchor alignment

In [ ]:
# ---- run stage 1: clean + anchor alignment (seconds) ---------------
articles = []          # per-article cleaned sentence lists
anchor_rows = []       # stage-1 pairs (confident + unsure)
unmatched_items = []   # (kus_title, kus_pos, sentence)
tier_counts = collections.Counter()

for line in open(IN_PATH, encoding='utf-8'):
    rec = json.loads(line)
    kus_sents = split_sentences(deglue_kus(normalize(rec['kus_text'])), 'kus')
    en_sents = split_sentences(normalize(rec['en_text']), 'en')
    articles.append({'kus_title': rec['kus_title'], 'en_title': rec['en_title'],
                     'kus_sents': kus_sents, 'en_sents': en_sents})
    if not kus_sents or not en_sents:
        unmatched_items += [(rec['kus_title'], i, s) for i, s in enumerate(kus_sents)]
        continue
    rows, unmatched = align_article(kus_sents, en_sents)
    for r in rows:
        r['kus_title'] = rec['kus_title']
        r['en_title'] = rec['en_title']
        tier_counts[r['tier']] += 1
    anchor_rows += rows
    unmatched_items += [(rec['kus_title'], ki, kus_sents[ki]) for ki in unmatched]

cols = ['tier', 'score', 'margin', 'mutual', 'anchors', 'kus_title', 'kus_pos',
        'kus_sentence', 'en_title', 'en_pos', 'en_sentence', 'verdict']
with open(f'{OUT_DIR}/anchor_pairs.csv', 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
    w.writeheader()
    for r in anchor_rows:
        w.writerow({**r, 'verdict': ''})
with open(f'{OUT_DIR}/unmatched_kus.csv', 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(['kus_title', 'kus_pos', 'kus_sentence'])
    w.writerows(unmatched_items)

print(f'articles={len(articles)}  confident={tier_counts["confident"]}  '
      f'unsure={tier_counts["unsure"]}  unmatched={len(unmatched_items)}  '
      f'deglue_fixes={deglue_count}')


## Stage 2a — translate with tekyerema-nllb600m-v1 (~25–40 min)

In [ ]:
# ---- stage 2a: translate every kusaal sentence (the long cell) -----
# ~25-40 min on a T4. Checkpoints to Drive: if Colab disconnects,
# just reconnect and run all cells again - finished work is skipped.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_ID = 'PrinceAlhassanNasamu/tekyerema-nllb600m-v1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID, token=hf_token, torch_dtype=torch.float16).to('cuda').eval()
tokenizer.src_lang = 'kus_Latn'
ENG_ID = tokenizer.convert_tokens_to_ids('eng_Latn')

kus_items = [(a['kus_title'], i, s)
             for a in articles for i, s in enumerate(a['kus_sents'])]
kus_index = {(t, p): j for j, (t, p, _) in enumerate(kus_items)}

CKPT = f'{OUT_DIR}/translations.jsonl'
done = {}
if os.path.exists(CKPT):
    for line in open(CKPT, encoding='utf-8'):
        try:
            d = json.loads(line)
            done[(d['kus_title'], d['kus_pos'])] = d['mt']
        except json.JSONDecodeError:
            pass
translations = [None] * len(kus_items)
todo = []
for j, (t, p, _) in enumerate(kus_items):
    if (t, p) in done:
        translations[j] = done[(t, p)]
    else:
        todo.append(j)
print(f'{len(done)} already translated, {len(todo)} to go')

B = 64
order = sorted(todo, key=lambda i: len(kus_items[i][2]))
t0 = time.time()
with open(CKPT, 'a', encoding='utf-8') as ckpt:
    for bi in range(0, len(order), B):
        idxs = order[bi:bi + B]
        enc = tokenizer([kus_items[i][2] for i in idxs], return_tensors='pt',
                        padding=True, truncation=True, max_length=256).to('cuda')
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=ENG_ID,
                                 num_beams=4, max_new_tokens=160)
        for i, o in zip(idxs, tokenizer.batch_decode(gen, skip_special_tokens=True)):
            translations[i] = o.strip()
            t, p, _ = kus_items[i]
            ckpt.write(json.dumps({'kus_title': t, 'kus_pos': p, 'mt': o.strip()},
                                  ensure_ascii=False) + '\n')
        ckpt.flush()
        if (bi // B) % 10 == 0:
            el = time.time() - t0
            done_n = bi + len(idxs)
            print(f'{done_n}/{len(order)}  {el/60:.1f} min  '
                  f'({done_n/max(el,1):.1f} sent/s)', flush=True)
print(f'translation done in {(time.time() - t0)/60:.1f} min')


## Stage 2b — verify, mine, write final CSV

In [ ]:
# ---- stage 2b: verify + mine + final csv ---------------------------
import random
import numpy as np
import sacrebleu
from sentence_transformers import SentenceTransformer

del model
torch.cuda.empty_cache()

en_items, en_range, en_title_of = [], {}, {}
for a in articles:
    start = len(en_items)
    en_items.extend(a['en_sents'])
    en_range[a['kus_title']] = (start, len(en_items))
    en_title_of[a['kus_title']] = a['en_title']

emb_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2',
                                device='cuda')
mt_emb = emb_model.encode(translations, batch_size=256,
                          normalize_embeddings=True, show_progress_bar=True)
en_emb = emb_model.encode(en_items, batch_size=256,
                          normalize_embeddings=True, show_progress_bar=True)

def chrf(hyp, ref):
    return sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score

def verdict_of(cos, ch, margin=None):
    if cos >= 0.65 or (cos >= 0.55 and ch >= 45):
        v = 'accept'
    elif cos < 0.40 and ch < 25:
        v = 'reject'
    else:
        v = 'uncertain'
    if margin is not None and v == 'accept' and margin < 0.04:
        v = 'uncertain'
    return v

out_rows, conf_cos = [], []
for r in anchor_rows:
    ki = kus_index[(r['kus_title'], r['kus_pos'])]
    start, _ = en_range[r['kus_title']]
    ei = start + r['en_pos']
    cos = float(np.dot(mt_emb[ki], en_emb[ei]))
    ch = chrf(translations[ki], en_items[ei])
    if r['tier'] == 'confident':
        conf_cos.append(cos)
    out_rows.append({'origin': 'anchor-' + r['tier'],
                     'verdict': verdict_of(cos, ch),
                     'cos': round(cos, 4), 'chrf': round(ch, 1), 'margin': '',
                     'anchor_score': r['score'],
                     'kus_title': r['kus_title'], 'kus_pos': r['kus_pos'],
                     'kus_sentence': r['kus_sentence'],
                     'machine_translation': translations[ki],
                     'en_title': r['en_title'], 'en_pos': r['en_pos'],
                     'en_sentence': r['en_sentence']})

for title, pos, sent in unmatched_items:
    ki = kus_index[(title, pos)]
    start, end = en_range[title]
    if end <= start:
        continue
    sims = en_emb[start:end] @ mt_emb[ki]
    top = int(np.argmax(sims))
    best = float(sims[top])
    second = float(np.partition(sims, -2)[-2]) if end - start > 1 else 0.0
    ei = start + top
    ch = chrf(translations[ki], en_items[ei])
    out_rows.append({'origin': 'mined',
                     'verdict': verdict_of(best, ch, best - second),
                     'cos': round(best, 4), 'chrf': round(ch, 1),
                     'margin': round(best - second, 4), 'anchor_score': '',
                     'kus_title': title, 'kus_pos': pos, 'kus_sentence': sent,
                     'machine_translation': translations[ki],
                     'en_title': en_title_of[title], 'en_pos': top,
                     'en_sentence': en_items[ei]})

rng = random.Random(7)
neg_cos = []
titles = list(en_range)
for _ in range(5000):
    ki = rng.randrange(len(kus_items))
    t = rng.choice(titles)
    while t == kus_items[ki][0]:
        t = rng.choice(titles)
    s, e = en_range[t]
    if e > s:
        neg_cos.append(float(np.dot(mt_emb[ki], en_emb[rng.randrange(s, e)])))

pct = lambda a, q: round(float(np.percentile(a, q)), 3)
counts = collections.Counter((r['origin'], r['verdict']) for r in out_rows)
lines = [
    f'anchor-confident cosine  p5={pct(conf_cos,5)} p25={pct(conf_cos,25)} '
    f'p50={pct(conf_cos,50)} p90={pct(conf_cos,90)}',
    f'random-negative cosine   p50={pct(neg_cos,50)} p95={pct(neg_cos,95)} '
    f'p99={pct(neg_cos,99)}',
    '', 'counts (origin, verdict):',
] + [f'  {o:18s} {v:10s} {n}' for (o, v), n in sorted(counts.items())]
report = '\n'.join(lines)
print(report)
with open(f'{OUT_DIR}/stage2_stats.txt', 'w', encoding='utf-8') as f:
    f.write(report + '\n')

cols = ['origin', 'verdict', 'cos', 'chrf', 'margin', 'anchor_score',
        'kus_title', 'kus_pos', 'kus_sentence', 'machine_translation',
        'en_title', 'en_pos', 'en_sentence']
with open(f'{OUT_DIR}/stage2_pairs.csv', 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
    w.writeheader()
    w.writerows(out_rows)
print(f'wrote {len(out_rows)} rows to {OUT_DIR}/stage2_pairs.csv')
print('DONE')
